# Causal Machine Learning — Iron Supplementation Decision Tool

**4 outcomes:** Hemoglobina (beneficio), Crescimento/WFL (beneficio), Escore infeccioso (dano), Diarreia (dano)

**Pipeline:** CausalForestDML + T-Learner + S-Learner | Holdout 80/20 | Bootstrap CI n=1000 | GATES | SHAP | Equidade | Deploy

---

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import norm, pearsonr, spearmanr
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

print('Libs OK')

## 1. Dados

In [ ]:
DF = pd.read_csv('/Users/marcelocarvalhoesilva/project/iron/data_crianca_calib_anon.csv', low_memory=False)
DF['age_months'] = DF['b05a_idade_em_meses'].str.extract(r'(\d+)').astype(float)
DF = DF[(DF['age_months'] >= 6) & (DF['age_months'] <= 18)].copy()
DF['iron_any'] = (DF['vd_supl1_com_ferro'] == 'Sim').astype(int)

# Confounders
DF['male'] = (DF['b02_sexo'] == 'Masculino').astype(int)
DF['race_preta'] = (DF['d01_cor'] == 'Preta').astype(int)
DF['race_parda'] = DF['d01_cor'].isin(['Parda (mulata, cabocla, cafuza, mameluca)', 'Parda']).astype(int)
for r, code in [('Norte', 1), ('Nordeste', 2), ('Sul', 4)]:
    DF[f'reg_{r.lower()}'] = (DF['a00_regiao'] == code).astype(int)
DF['reg_co'] = (DF['a00_regiao'] == 5).astype(int)
DF['urban'] = (DF['a11_situacao'] == 'Urbano').astype(int) if DF['a11_situacao'].dtype == 'object' else (DF['a11_situacao'] == 1).astype(int)
DF['educ_raw'] = pd.to_numeric(DF['j10_serie'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
DF['educ_fund_comp'] = ((DF['educ_raw'] >= 9) & (DF['educ_raw'] < 12)).astype(int)
DF['educ_medio'] = (DF['educ_raw'] == 12).astype(int)
DF['educ_superior'] = (DF['educ_raw'] > 12).astype(int)
DF['ien'] = pd.to_numeric(DF['vd_ien_quintos'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
DF['ebia_leve'] = (DF['vd_ebia_categ'] == 'Insegurança leve').astype(int) if DF['vd_ebia_categ'].dtype == 'object' else 0
DF['ebia_mod'] = (DF['vd_ebia_categ'] == 'Insegurança moderada').astype(int) if DF['vd_ebia_categ'].dtype == 'object' else 0
DF['ebia_grave'] = (DF['vd_ebia_categ'] == 'Insegurança grave').astype(int) if DF['vd_ebia_categ'].dtype == 'object' else 0
DF['esgoto_adeq'] = DF['p10_esgoto'].isin(['Rede geral de esgoto ou pluvial', 'Fossa séptica ligada a rede', 'Fossa séptica não ligada a rede', 1, 2, 3]).astype(int)
DF['agua_rede'] = DF['p11_agua'].isin(['Rede geral de distribuição', 1]).astype(int)
DF['cesarean'] = DF['h04_parto'].isin(['Cesariana agendada (eletiva)', 'Cesariana de urgência (não agendada)', 2, 3]).astype(int)
DF['gest_weeks'] = pd.to_numeric(DF['h01_semanas_gravidez'], errors='coerce')
DF['birth_weight'] = pd.to_numeric(DF['h02_peso'], errors='coerce')
DF['prenatal_visits'] = pd.to_numeric(DF['k05_prenatal_consultas'], errors='coerce')
DF['prenatal_visits'] = DF['prenatal_visits'].fillna(DF['prenatal_visits'].median())
DF['maternal_age'] = pd.to_numeric(DF['bb04_idade_da_mae'], errors='coerce')
DF['breastfed'] = (DF['e01_leite_peito'] == 'Sim').astype(int) if DF['e01_leite_peito'].dtype == 'object' else (DF['e01_leite_peito'] == 1).astype(int)
DF['formula'] = (DF['e10_formula_infantil'] == 'Sim').astype(int) if DF['e10_formula_infantil'].dtype == 'object' else (DF['e10_formula_infantil'] == 1).astype(int)
DF['cow_milk'] = ((DF['e06_leite_vaca_po'].isin(['Sim', 1])) | (DF['e07_leite_vaca_liquido'].isin(['Sim', 1]))).astype(int)
DF['feed_exclusive'] = ((DF['breastfed'] == 1) & (DF['formula'] == 0) & (DF['cow_milk'] == 0)).astype(int)
DF['feed_mixed'] = ((DF['breastfed'] == 1) & ((DF['formula'] == 1) | (DF['cow_milk'] == 1))).astype(int)

# Morbidade
DF['diarreia_bin'] = DF['h13_diarreia'].map({'Sim': 1, 'Não': 0})
DF['febre_bin'] = DF['h19_febre'].map({'Sim': 1, 'Não': 0})
DF['tosse_bin'] = DF['h14_tosse'].map({'Sim': 1, 'Não': 0})
DF['resp_dificil_bin'] = DF['h15_respiracao'].map({'Sim': 1, 'Não': 0})
DF['intern_resp_bin'] = DF['h211_internado_respiratoria'].map({'Sim': 1, 'Não': 0})
DF['intern_intest_bin'] = DF['h212_internado_intestinais'].map({'Sim': 1, 'Não': 0})
DF['infection_score'] = (DF['diarreia_bin'].fillna(0) + DF['febre_bin'].fillna(0) + DF['tosse_bin'].fillna(0) + DF['resp_dificil_bin'].fillna(0) + DF['intern_resp_bin'].fillna(0) + DF['intern_intest_bin'].fillna(0))

print(f'N: {len(DF)} | Ferro: {DF["iron_any"].sum()} ({100*DF["iron_any"].mean():.1f}%)')
print(f'Infection score: {DF["infection_score"].describe().to_dict()}')

## 2. Features e Outcomes

In [ ]:
FEATURES = [
    'age_months', 'male', 'race_preta', 'race_parda',
    'reg_norte', 'reg_nordeste', 'reg_sul', 'reg_co', 'urban',
    'educ_fund_comp', 'educ_medio', 'educ_superior', 'ien',
    'ebia_leve', 'ebia_mod', 'ebia_grave',
    'esgoto_adeq', 'agua_rede', 'cesarean',
    'gest_weeks', 'birth_weight', 'prenatal_visits', 'maternal_age',
    'breastfed', 'formula', 'feed_exclusive', 'feed_mixed',
]

FEATURE_NAMES = [
    'Idade (meses)', 'Sexo masculino', 'Cor preta', 'Cor parda',
    'Norte', 'Nordeste', 'Sul', 'Centro-Oeste', 'Urbano',
    'Educ: Fund.completo', 'Educ: Medio', 'Educ: Superior', 'IEN (quintil)',
    'EBIA: leve', 'EBIA: moderada', 'EBIA: grave',
    'Saneamento adequado', 'Agua rede', 'Cesariana',
    'Idade gestacional', 'Peso nascer', 'Consultas PN', 'Idade materna',
    'Aleitamento materno', 'Formula infantil', 'Aleit. exclusivo', 'Aleit. misto',
]

OUTCOMES = {
    'vd_hb_final':     ('Hemoglobina (g/dL)',             'benefit'),
    'vd_anthro_zwfl':  ('Crescimento WFL (z-score)',      'benefit'),
    'infection_score': ('Escore morbidade infecciosa',    'harm'),
    'diarreia_bin':    ('Diarreia 15 dias',               'harm'),
}

print(f'Features: {len(FEATURES)} | Outcomes: {len(OUTCOMES)}')
for col, (name, d) in OUTCOMES.items():
    print(f'  {name} ({d})')

## 3. Train-Test Split + Treinar CausalForest (4 outcomes)

In [ ]:
results = {}

for outcome_col, (outcome_name, direction) in OUTCOMES.items():
    print(f"\n{'='*70}")
    print(f'{outcome_name} ({direction})')
    print(f"{'='*70}")
    
    data = DF[['iron_any', outcome_col] + FEATURES].dropna().copy()
    train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=42, stratify=data['iron_any'])
    train, test = data.loc[train_idx], data.loc[test_idx]
    
    Y_tr, T_tr, X_tr = train[outcome_col].values, train['iron_any'].values, train[FEATURES].values
    Y_te, T_te, X_te = test[outcome_col].values, test['iron_any'].values, test[FEATURES].values
    
    print(f'  Train: {len(train)} | Test: {len(test)} | Overlap: {len(set(train_idx)&set(test_idx))} | Iron diff: {abs(train["iron_any"].mean()-test["iron_any"].mean())*100:.2f}%')
    
    # CausalForest
    cf = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
        model_t=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42),
        n_estimators=500, min_samples_leaf=30, random_state=42, verbose=0
    )
    cf.fit(Y_tr, T_tr, X=X_tr)
    cate_tr = cf.effect(X_tr).flatten()
    cate_te = cf.effect(X_te).flatten()
    
    # T-Learner
    tl = TLearner(models=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42))
    tl.fit(Y_tr, T_tr, X=X_tr)
    
    # S-Learner
    sl = SLearner(overall_model=GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42))
    sl.fit(Y_tr, T_tr, X=X_tr)
    
    print(f'  ATE train: {cate_tr.mean():.4f} | ATE test: {cate_te.mean():.4f} | Diff: {abs(cate_tr.mean()-cate_te.mean()):.4f}')
    print(f'  % CATE>0 (test): {(cate_te>0).mean()*100:.1f}% | % CATE<0: {(cate_te<0).mean()*100:.1f}%')
    
    # Concordancia
    tl_te = tl.effect(X_te).flatten()
    sl_te = sl.effect(X_te).flatten()
    r_cf_tl, _ = pearsonr(cate_te, tl_te)
    r_cf_sl, _ = pearsonr(cate_te, sl_te)
    print(f'  Concordancia test: CF vs TL r={r_cf_tl:.3f} | CF vs SL r={r_cf_sl:.3f}')
    
    results[outcome_col] = {
        'data': data, 'train': train, 'test': test, 'cf': cf,
        'cate_tr': cate_tr, 'cate_te': cate_te, 'name': outcome_name, 'direction': direction
    }

## 4. Bootstrap CI (n=1000) por Subgrupo — Todos os Outcomes

In [ ]:
def boot_ci(vals, n=1000):
    means = [np.random.choice(vals, len(vals), replace=True).mean() for _ in range(n)]
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

for oc, r in results.items():
    print(f"\n{'='*80}")
    print(f"BOOTSTRAP CI — {r['name']}")
    print(f"{'='*80}")
    
    test = r['test'].copy()
    test['cate'] = r['cate_te']
    
    subgroups = {
        'ATE Global': test.index > -1,
        'Nao amamentado': test['breastfed'] == 0,
        'Amamentado': test['breastfed'] == 1,
        'Aleit. exclusivo': test['feed_exclusive'] == 1,
        'Idade 6-11m': test['age_months'] <= 11,
        'Idade 12-18m': test['age_months'] > 11,
        'Nao amam 6-11m': (test['breastfed']==0) & (test['age_months']<=11),
        'Nao amam 12-18m': (test['breastfed']==0) & (test['age_months']>11),
        'Amam 6-11m': (test['breastfed']==1) & (test['age_months']<=11),
        'Amam 12-18m': (test['breastfed']==1) & (test['age_months']>11),
        'IEN Q1': test['ien'] == 1,
        'IEN Q5': test['ien'] == 5,
    }
    
    print(f"  {'Subgrupo':<25} {'N':>5} {'CATE':>8} {'95% CI':>22} {'Sig':>4}")
    print(f"  {'-'*64}")
    for name, mask in subgroups.items():
        if mask.sum() < 10: continue
        vals = test.loc[mask, 'cate'].values
        lo, hi = boot_ci(vals)
        sig = '*' if (lo > 0 or hi < 0) else ''
        print(f"  {name:<25} {mask.sum():>5} {vals.mean():>8.4f} ({lo:>8.4f} to {hi:>8.4f}) {sig:>4}")

## 5. GATES — Heterogeneidade Confirmada?

In [ ]:
for oc, r in results.items():
    print(f"\n{'='*70}")
    print(f"GATES — {r['name']}")
    print(f"{'='*70}")
    test = r['test'].copy()
    test['cate'] = r['cate_te']
    test['cate_q'] = pd.qcut(test['cate'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
    
    for q in ['Q1','Q2','Q3','Q4','Q5']:
        sub = test[test['cate_q']==q]
        print(f"  {q}: N={len(sub):>4} | CATE={sub['cate'].mean():>8.4f} | Y_ferro={sub.loc[sub['iron_any']==1, oc].mean():>7.3f} | Y_nao={sub.loc[sub['iron_any']==0, oc].mean():>7.3f}")
    
    q1v = test.loc[test['cate_q']=='Q1','cate'].values
    q5v = test.loc[test['cate_q']=='Q5','cate'].values
    diffs = [np.random.choice(q5v,len(q5v),True).mean()-np.random.choice(q1v,len(q1v),True).mean() for _ in range(1000)]
    lo, hi = np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)
    sig = 'CONFIRMADA' if (lo>0 or hi<0) else 'nao significativa'
    print(f"  Q5-Q1: {np.mean(diffs):.4f} (CI: {lo:.4f} to {hi:.4f}) — Heterogeneidade {sig}")

## 6. SHAP (surrogate) — 4 Outcomes

In [ ]:
import os
os.makedirs('/Users/marcelocarvalhoesilva/project/iron/data_Eval/figures', exist_ok=True)

for oc, r in results.items():
    print(f"\n{'='*70}")
    print(f"SHAP — {r['name']}")
    print(f"{'='*70}")
    
    data = r['data']
    X_all = data[FEATURES].values
    cate_all = r['cf'].effect(X_all).flatten()
    
    surrogate = GradientBoostingRegressor(n_estimators=300, max_depth=4, random_state=42)
    surrogate.fit(X_all, cate_all)
    print(f"  Surrogate R2: {surrogate.score(X_all, cate_all):.4f}")
    
    explainer = shap.TreeExplainer(surrogate)
    sv = explainer.shap_values(X_all)
    
    imp = np.abs(sv).mean(axis=0)
    ranking = sorted(zip(FEATURE_NAMES, imp), key=lambda x: -x[1])
    print(f"  Top 5: {', '.join([f'{n}({v:.4f})' for n,v in ranking[:5]])}")
    
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(sv, features=X_all, feature_names=FEATURE_NAMES, show=False, max_display=15)
    short = oc.replace('vd_','').replace('_final','').replace('_bin','')
    plt.title(f'SHAP — {r["name"]}', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'/Users/marcelocarvalhoesilva/project/iron/data_Eval/figures/shap_{short}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    r['surrogate'] = surrogate
    r['shap_vals'] = sv

## 7. Equidade com Bootstrap CI

In [ ]:
for oc, r in results.items():
    print(f"\n{'='*70}")
    print(f"EQUIDADE — {r['name']}")
    print(f"{'='*70}")
    data = r['data'].copy()
    data['cate'] = r['cf'].effect(data[FEATURES].values).flatten()
    
    groups = {
        'IEN Q1': data['ien']==1, 'IEN Q2': data['ien']==2, 'IEN Q3': data['ien']==3,
        'IEN Q4': data['ien']==4, 'IEN Q5': data['ien']==5,
        'Cor preta': data['race_preta']==1, 'Cor parda': data['race_parda']==1,
        'Cor branca/outra': (data['race_preta']==0)&(data['race_parda']==0),
        'Inseg. grave': data['ebia_grave']==1, 'Seguranca alim.': (data['ebia_leve']==0)&(data['ebia_mod']==0)&(data['ebia_grave']==0),
        'Saneam. adequado': data['esgoto_adeq']==1, 'Saneam. inadequado': data['esgoto_adeq']==0,
    }
    print(f"  {'Grupo':<25} {'N':>6} {'CATE':>8} {'95% CI':>22} {'Sig':>4}")
    print(f"  {'-'*65}")
    for name, mask in groups.items():
        if mask.sum() < 10: continue
        vals = data.loc[mask, 'cate'].values
        lo, hi = boot_ci(vals)
        sig = '*' if (lo>0 or hi<0) else ''
        print(f"  {name:<25} {mask.sum():>6} {vals.mean():>8.4f} ({lo:>8.4f} to {hi:>8.4f}) {sig:>4}")

## 8. Decisao Clinica (4 outcomes) + Scatter

In [ ]:
# Juntar CATEs
merged = DF.copy()
for oc, r in results.items():
    data = r['data']
    X = data[FEATURES].values
    cate = r['cf'].effect(X).flatten()
    short = oc.replace('vd_','').replace('_final','').replace('_bin','')
    merged.loc[data.index, f'cate_{short}'] = cate

cate_cols = [f'cate_{oc.replace("vd_","").replace("_final","").replace("_bin","")}' for oc in OUTCOMES]
available = [c for c in cate_cols if c in merged.columns]
sc = merged.dropna(subset=available).copy()
print(f'N com todos os CATEs: {len(sc)}')

# Decisao
def decide(row):
    hb = row.get('cate_hb', 0)
    wfl = row.get('cate_anthro_zwfl', 0)
    inf = row.get('cate_infection_score', 0)
    dia = row.get('cate_diarreia', 0)
    infection_risk = (inf > 0.05) or (dia > 0.02)
    hb_benefit = hb > 0.01
    growth_harm = wfl < -0.05
    if hb_benefit and not infection_risk and not growth_harm:
        return 'SUPLEMENTAR'
    elif hb_benefit and (infection_risk or growth_harm):
        return 'TRADE-OFF'
    else:
        return 'NAO SUPLEMENTAR'

sc['decision'] = sc.apply(decide, axis=1)
print(f'\n{sc["decision"].value_counts()}')
print(f'\nPor aleitamento:')
print(pd.crosstab(sc['breastfed'].map({0:'Nao amam',1:'Amamentado'}), sc['decision']))
print(f'\nPor aleitamento x idade:')
sc['bf_age'] = sc.apply(lambda r: f"{'Amam' if r['breastfed']==1 else 'Nao amam'} {'6-11m' if r['age_months']<=11 else '12-18m'}", axis=1)
print(pd.crosstab(sc['bf_age'], sc['decision']))

# Scatter 3 paineis
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Decisao: Hb vs Infeccao
colors = {'SUPLEMENTAR':'green', 'TRADE-OFF':'orange', 'NAO SUPLEMENTAR':'red'}
for dec, color in colors.items():
    sub = sc[sc['decision']==dec]
    if len(sub)>0:
        axes[0].scatter(sub['cate_hb'], sub['cate_infection_score'], alpha=0.4, c=color, label=f'{dec} ({len(sub)})', s=20)
axes[0].axhline(0,color='gray',ls='--',alpha=0.5); axes[0].axvline(0,color='gray',ls='--',alpha=0.5)
axes[0].set_xlabel('CATE Hemoglobina'); axes[0].set_ylabel('CATE Escore Infeccioso')
axes[0].set_title('Decisao Clinica'); axes[0].legend(fontsize=8)

# 2. Aleitamento: Hb vs Infeccao
for bf, color, label in [(0,'red','Nao amam'),(1,'blue','Amamentado')]:
    sub = sc[sc['breastfed']==bf]
    axes[1].scatter(sub['cate_hb'], sub['cate_infection_score'], alpha=0.3, c=color, label=label, s=20)
axes[1].axhline(0,color='gray',ls='--',alpha=0.5); axes[1].axvline(0,color='gray',ls='--',alpha=0.5)
axes[1].set_xlabel('CATE Hemoglobina'); axes[1].set_ylabel('CATE Escore Infeccioso')
axes[1].set_title('Por Aleitamento'); axes[1].legend(fontsize=10)

# 3. Hb vs WFL colorido por infeccao
if 'cate_infection_score' in sc.columns:
    s = axes[2].scatter(sc['cate_hb'], sc['cate_anthro_zwfl'], c=sc['cate_infection_score'], cmap='RdYlGn_r', alpha=0.5, s=20)
    plt.colorbar(s, ax=axes[2], label='CATE Infeccao')
axes[2].axhline(0,color='gray',ls='--',alpha=0.5); axes[2].axvline(0,color='gray',ls='--',alpha=0.5)
axes[2].set_xlabel('CATE Hemoglobina'); axes[2].set_ylabel('CATE Crescimento (WFL)')
axes[2].set_title('Hb vs Crescimento (cor=Infeccao)')

plt.suptitle('Decisao Individualizada: Beneficio vs Risco Infeccioso vs Crescimento', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/Users/marcelocarvalhoesilva/project/iron/data_Eval/figures/clinical_decision.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Deploy — Exportar 4 Modelos para JS (GitHub Pages)

In [ ]:
import m2cgen as m2c
import json, os, joblib

deploy_dir = '/Users/marcelocarvalhoesilva/project/iron/deploy'
os.makedirs(deploy_dir, exist_ok=True)

for oc, r in results.items():
    data = r['data']
    X = data[FEATURES].values
    cate = r['cf'].effect(X).flatten()
    
    # RF surrogate para JS
    rf = RandomForestRegressor(n_estimators=30, max_depth=5, random_state=42)
    rf.fit(X, cate)
    r2 = rf.score(X, cate)
    
    short = oc.replace('vd_','').replace('_final','').replace('_bin','')
    
    js = m2c.export_to_javascript(rf)
    js = js.replace('function score(', f'function score_{short}(')
    
    with open(f'{deploy_dir}/model_{short}.js', 'w') as f:
        f.write(js)
    
    joblib.dump(rf, f'{deploy_dir}/surrogate_{short}.joblib')
    print(f'{r["name"]}: R2={r2:.4f} | JS={len(js)/1024:.0f}KB | score_{short}()')

# Config
config = {
    'features': FEATURES,
    'feature_names': FEATURE_NAMES,
    'outcomes': {oc.replace('vd_','').replace('_final','').replace('_bin',''): {'name': n, 'direction': d} for oc,(n,d) in OUTCOMES.items()},
}
with open(f'{deploy_dir}/config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f'\nDeploy pronto em {deploy_dir}/')